# Current `code_*` MCP architecture and audit

**Audit date:** 2026-08-26  
**Source HEAD:** `0595e15eff7e1e4a00dbab50fb9678b434cc04ac`  
**Indexed graph HEAD:** `8a5a7e3457e6f823c7128193f8a844ffb06636e2`  
**Issue:** `bd-1ykp`

## Executive conclusion

The dirty/stale-graph happy path is **correctly fail-closed**: it creates an exact start snapshot, serves the query from one immutable overlay generation, obtains an exact end snapshot, and discards the result if the identities differ. The persistent generation eliminates repeated merge/filter/sort/dedup work.

The architecture still violates its performance and metadata contracts in four important ways:

1. `code_symbol_search` / legacy `code_search` retains a separate legacy metadata-preflight route while the other eight tools use the direct generation route.
2. Validation is scoped to each MCP request, not to a multi-tool journey, so `search → read → callers` repeats the exact start/end lease for every call.
3. The “two observations” are expensive: each snapshot contains `rev-parse → exact status → rev-parse`; the primary route therefore performs six Git subprocesses per request.
4. Certified overlay metadata forces `worktree_dirty=false`, even when indexed and worktree HEADs differ. That makes freshness and physical Git dirtiness mean the same field, contradicting the public metadata contract.

**Bottom line:** source-of-truth safety is strong; individual warm `code_*` under 10 ms is not achievable with the current per-request exact observer. Cross-request batching or a supported synchronized Git token/barrier is required.

## Scope and evidence discipline

This is an audit of the **current source tree**, not a restatement of the older graph artifact.

- Orientation came from `knowledge_context_pack_2`; every behavioral claim was then grounded with overlay-aware `code_read_symbol`.
- Exact source reads reported `source_origin=worktree` and `stale=true`, which is expected because the graph artifact trails HEAD.
- Native reads were used only for current raw bytes, Git state, and benchmark JSON after the graph tools established the relevant symbols.
- The 30-run matrix is from `.spur/bench-evidence/task6-overlay-generation-matrix.json`.
- Live MCP timings are wall-clock samples from the currently running server. They are labeled separately because the process build SHA is not exposed, so they are not proof that the binary equals source HEAD.

### Trust model

The graph artifact supplies the stable base. The exact overlay supplies all committed, tracked, untracked, renamed, and deleted supported-language changes relative to that base. The response is authoritative only while the start and end `SnapshotIdentity` values match.

## Tool surface and route partition

The module dispatches nine public tools. Eight use `code_graph_backend_response_with_refresh`; search remains a special path.

| Public tool | Handler wrapper | Auto overlay entry | Source support |
|---|---|---|---|
| `code_symbol_search` / `code_search` | `code_search_response` | Legacy metadata preflight, then `attempt_refresh` | No |
| `code_resolve` | generic wrapper | Direct before legacy preflight | No |
| `code_file_symbols` | generic wrapper | Direct before legacy preflight | No |
| `code_symbol_info` | generic wrapper | Direct before legacy preflight | No |
| `code_read_symbol` | generic wrapper allowing source | Direct before legacy preflight | Yes |
| `code_callers` | generic wrapper | Direct before legacy preflight | No |
| `code_callees` | generic wrapper | Direct before legacy preflight | No |
| `code_subgraph` | generic wrapper | Direct before legacy preflight | No |
| `code_symbol_history` | generic wrapper | Direct before legacy preflight | No |

The next cell formally checks that this route partition is total, deterministic, and exclusive. It does **not** prove that the two routes have equivalent cost or metadata semantics.

In [ ]:
flowchart LR
    SPEC["`@spec CODE-MCP-ROUTE
@type Tool = enum[symbol_search, resolve, file_symbols, symbol_info, read_symbol, callers, callees, subgraph, history]
@type Route = enum[legacy_preflight, direct_generation]
@input tool: Tool
@output route: Route
@requires VALID_TOOL: true`"]
    SEARCH["`@branch SEARCH_EXCEPTION
@when tool = symbol_search
@ensures SEARCH_ROUTE: route = legacy_preflight`"]
    GENERIC["`@branch GENERIC_ROUTE
@when tool = resolve or tool = file_symbols or tool = symbol_info or tool = read_symbol or tool = callers or tool = callees or tool = subgraph or tool = history
@ensures GENERIC_RESULT: route = direct_generation`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify CONSISTENT: witness consistency`"]
    SPEC --> SEARCH --> CHECK
    SPEC --> GENERIC --> CHECK

## Verified component architecture

This NS-Mermaid `architecture_policy` v1 view binds the current component topology into three explicit zones:

- **Caller** — the MCP client.
- **MCP edge** — `GraphMcpModule` dispatch.
- **Graph and worktree** — base graph storage, exact Git observation, overlay caches/generations, pinned query execution, and certified metadata.

The two known architectural findings are visible in the topology: **F1** is the separate legacy search preflight branch, while **F2** is the per-request exact Git observation lease. Formal scope is deliberately limited to topology completeness, non-vacuity, resolved edges, and zone-policy conformance; it does not invent deployment capacity, network security, or runtime-latency guarantees.

In [ ]:
architecture-beta
    group public(cloud)[Caller]
    group edge(cloud)[MCP edge]
    group private(cloud)[Graph and worktree]

    service client(internet)[MCP client] in public
    service router(server)[GraphMcpModule router] in edge

    service base(database)[Parquet or memory base] in private
    service snapshot(database)[Snapshot cache] in private
    service observer(server)[Per request exact Git lease F2] in private
    service git(disk)[Git metadata] in private
    service files(disk)[Worktree files] in private
    service delta(database)[Exact delta cache] in private
    service generation(database)[Persistent overlay generation] in private
    service replay(database)[Per request replay] in private
    service pinned(server)[Pinned generation client] in private
    service query(server)[code tool handlers] in private
    service metadata(server)[Certified metadata] in private
    service search(server)[Legacy search preflight F1] in private

    client:R --> L:router
    router:R --> L:base
    router:B --> T:snapshot
    snapshot:R --> L:observer
    observer:R --> L:git
    observer:R --> L:files
    snapshot:B --> T:delta
    delta:R --> L:generation
    base:B --> T:generation
    router:B --> T:replay
    replay:R --> L:pinned
    generation:R --> L:pinned
    pinned:R --> L:query
    query:R --> L:metadata
    router:R --> L:search
    search:B --> T:observer
    search:R --> L:base
    %% @ns-policy {"version":1,"services":[{"id":"client"},{"id":"router"},{"id":"base"},{"id":"snapshot"},{"id":"observer"},{"id":"git"},{"id":"files"},{"id":"delta"},{"id":"generation"},{"id":"replay"},{"id":"pinned"},{"id":"query"},{"id":"metadata"},{"id":"search"}],"edges":[{"from":"client","to":"router","protocol":"mcp","purpose":"request"},{"from":"router","to":"base","protocol":"in_process","purpose":"read_base"},{"from":"router","to":"snapshot","protocol":"in_process","purpose":"validate_snapshot"},{"from":"snapshot","to":"observer","protocol":"in_process","purpose":"observe_identity"},{"from":"observer","to":"git","protocol":"git_plumbing","purpose":"read_git_state"},{"from":"observer","to":"files","protocol":"filesystem","purpose":"read_worktree"},{"from":"snapshot","to":"delta","protocol":"in_process","purpose":"resolve_delta"},{"from":"delta","to":"generation","protocol":"in_process","purpose":"update_generation"},{"from":"base","to":"generation","protocol":"in_process","purpose":"seed_generation"},{"from":"router","to":"replay","protocol":"in_process","purpose":"reuse_request"},{"from":"replay","to":"pinned","protocol":"in_process","purpose":"pin_request"},{"from":"generation","to":"pinned","protocol":"in_process","purpose":"supply_generation"},{"from":"pinned","to":"query","protocol":"in_process","purpose":"execute_query"},{"from":"query","to":"metadata","protocol":"in_process","purpose":"certify_response"},{"from":"router","to":"search","protocol":"in_process","purpose":"legacy_search_preflight"},{"from":"search","to":"observer","protocol":"in_process","purpose":"validate_search"},{"from":"search","to":"base","protocol":"in_process","purpose":"search_base"}]}

## Primary dirty/stale-graph flow

For the eight generic tools with `overlay_fsmonitor_auto=true`:

1. Parse the response format and open the Parquet/in-memory graph backend.
2. Build an exact overlay snapshot from the worktree against the indexed graph base.
3. Reuse or extract the changed-path delta keyed by the exact snapshot identity.
4. Reuse the exact `OverlayGeneration`, or incrementally update the latest compatible generation.
5. Pin one immutable generation and execute every graph operation for that MCP request against it.
6. Re-observe the authoritative worktree identity after the query.
7. If identities match, attach certified metadata without response-file rescanning.
8. If they differ, discard the candidate result and execute the exact fallback path.

This is the strongest part of the design: a stale Parquet graph remains safe because the response is produced from **base + exact overlay**, not from the stale base alone.

In [ ]:
sequenceDiagram
    participant Client
    participant Router
    participant Backend
    participant Observer
    participant Generation
    participant Query
    participant Metadata
    Note over Client,Metadata: @spec CODE-MCP-HAPPY-PATH
    Client->>Router: request
    Note over Client,Router: @message REQUEST<br/>@from Client<br/>@to Router<br/>@event request<br/>@order 1<br/>@when true<br/>@ensures REQUESTED: true
    Router->>Backend: open_backend
    Note over Router,Backend: @message OPEN<br/>@from Router<br/>@to Backend<br/>@event open_backend<br/>@order 2<br/>@when true<br/>@ensures OPENED: true
    Router->>Observer: start_observe
    Note over Router,Observer: @message START_OBSERVE<br/>@from Router<br/>@to Observer<br/>@event start_observe<br/>@order 3<br/>@when true<br/>@ensures STARTED: true
    Observer->>Router: start_identity
    Note over Observer,Router: @message START_IDENTITY<br/>@from Observer<br/>@to Router<br/>@event start_identity<br/>@order 4<br/>@when true<br/>@ensures START_CERTIFIED: true
    Router->>Generation: get_generation
    Note over Router,Generation: @message GET_GENERATION<br/>@from Router<br/>@to Generation<br/>@event get_generation<br/>@order 5<br/>@when true<br/>@ensures GENERATION_REQUESTED: true
    Generation->>Router: pin_generation
    Note over Generation,Router: @message PIN<br/>@from Generation<br/>@to Router<br/>@event pin_generation<br/>@order 6<br/>@when true<br/>@ensures PINNED: true
    Router->>Query: execute_query
    Note over Router,Query: @message EXECUTE<br/>@from Router<br/>@to Query<br/>@event execute_query<br/>@order 7<br/>@when true<br/>@ensures EXECUTED: true
    Query->>Router: query_result
    Note over Query,Router: @message QUERY_RESULT<br/>@from Query<br/>@to Router<br/>@event query_result<br/>@order 8<br/>@when true<br/>@ensures RESULT_READY: true
    Router->>Observer: end_observe
    Note over Router,Observer: @message END_OBSERVE<br/>@from Router<br/>@to Observer<br/>@event end_observe<br/>@order 9<br/>@when true<br/>@ensures END_STARTED: true
    Observer->>Router: end_identity
    Note over Observer,Router: @message END_IDENTITY<br/>@from Observer<br/>@to Router<br/>@event end_identity<br/>@order 10<br/>@when true<br/>@ensures END_CERTIFIED: true
    Router->>Metadata: certify_metadata
    Note over Router,Metadata: @message CERTIFY<br/>@from Router<br/>@to Metadata<br/>@event certify_metadata<br/>@order 11<br/>@when true<br/>@ensures CERTIFIED: true
    Metadata->>Client: response
    Note over Metadata,Client: @message RESPONSE<br/>@from Metadata<br/>@to Client<br/>@event response<br/>@order 12<br/>@when true<br/>@ensures RESPONDED: true
    Note over Client,Metadata: @verify PROTOCOL: prove sequence_protocol

## What one “exact observation” actually does

`snapshot_with_runtime` is a bounded optimistic read:

- pre-`rev-parse HEAD`;
- filesystem identity of the Git index;
- `git status --porcelain=v2 -z --untracked-files=all`;
- index identity again, because status may refresh Git-owned index metadata;
- hash/stat only candidate changed paths, plus indexed-HEAD lag paths;
- post-`rev-parse HEAD` and final index identity;
- restat affected files;
- accept if all fences match, otherwise retry once, then use the filesystem oracle.

With current exact fallback capabilities, status explicitly sets `core.fsmonitor=false` and `core.untrackedCache=false`. Therefore one snapshot uses **three Git subprocesses**; the start/end request fence uses **six**. Index identity checks are filesystem metadata operations, not Git subprocesses.

In [ ]:
sequenceDiagram
    participant Snapshot
    participant Git
    participant Index
    participant Files
    Note over Snapshot,Files: @spec EXACT-GIT-OBSERVATION
    Snapshot->>Git: rev_parse_pre
    Note over Snapshot,Git: @message REV_PRE<br/>@from Snapshot<br/>@to Git<br/>@event rev_parse_pre<br/>@order 1<br/>@when true<br/>@ensures PRE_HEAD_REQUESTED: true
    Git->>Snapshot: pre_head
    Note over Git,Snapshot: @message PRE_HEAD<br/>@from Git<br/>@to Snapshot<br/>@event pre_head<br/>@order 2<br/>@when true<br/>@ensures PRE_HEAD_READY: true
    Snapshot->>Index: index_before
    Note over Snapshot,Index: @message INDEX_BEFORE<br/>@from Snapshot<br/>@to Index<br/>@event index_before<br/>@order 3<br/>@when true<br/>@ensures INDEX_BEFORE_READY: true
    Index->>Snapshot: index_before_value
    Note over Index,Snapshot: @message INDEX_BEFORE_VALUE<br/>@from Index<br/>@to Snapshot<br/>@event index_before_value<br/>@order 4<br/>@when true<br/>@ensures INDEX_BEFORE_OBSERVED: true
    Snapshot->>Git: exact_status
    Note over Snapshot,Git: @message STATUS<br/>@from Snapshot<br/>@to Git<br/>@event exact_status<br/>@order 5<br/>@when true<br/>@ensures STATUS_REQUESTED: true
    Git->>Snapshot: changed_entries
    Note over Git,Snapshot: @message ENTRIES<br/>@from Git<br/>@to Snapshot<br/>@event changed_entries<br/>@order 6<br/>@when true<br/>@ensures ENTRIES_READY: true
    Snapshot->>Index: index_after_status
    Note over Snapshot,Index: @message INDEX_AFTER_STATUS<br/>@from Snapshot<br/>@to Index<br/>@event index_after_status<br/>@order 7<br/>@when true<br/>@ensures INDEX_AFTER_READY: true
    Index->>Snapshot: index_after_value
    Note over Index,Snapshot: @message INDEX_AFTER_VALUE<br/>@from Index<br/>@to Snapshot<br/>@event index_after_value<br/>@order 8<br/>@when true<br/>@ensures INDEX_AFTER_OBSERVED: true
    Snapshot->>Files: hash_changed_paths
    Note over Snapshot,Files: @message HASH<br/>@from Snapshot<br/>@to Files<br/>@event hash_changed_paths<br/>@order 9<br/>@when true<br/>@ensures HASH_REQUESTED: true
    Files->>Snapshot: path_state
    Note over Files,Snapshot: @message PATH_STATE<br/>@from Files<br/>@to Snapshot<br/>@event path_state<br/>@order 10<br/>@when true<br/>@ensures PATHS_READY: true
    Snapshot->>Git: rev_parse_post
    Note over Snapshot,Git: @message REV_POST<br/>@from Snapshot<br/>@to Git<br/>@event rev_parse_post<br/>@order 11<br/>@when true<br/>@ensures POST_HEAD_REQUESTED: true
    Git->>Snapshot: post_head
    Note over Git,Snapshot: @message POST_HEAD<br/>@from Git<br/>@to Snapshot<br/>@event post_head<br/>@order 12<br/>@when true<br/>@ensures POST_HEAD_READY: true
    Snapshot->>Index: index_final
    Note over Snapshot,Index: @message INDEX_FINAL<br/>@from Snapshot<br/>@to Index<br/>@event index_final<br/>@order 13<br/>@when true<br/>@ensures FINAL_INDEX_REQUESTED: true
    Index->>Snapshot: final_identity
    Note over Index,Snapshot: @message FINAL_IDENTITY<br/>@from Index<br/>@to Snapshot<br/>@event final_identity<br/>@order 14<br/>@when true<br/>@ensures IDENTITY_READY: true
    Note over Snapshot,Files: @verify PROTOCOL: prove sequence_protocol

## Cache and incremental-generation architecture

| Layer | Key / scope | What is reused | Invalidated by |
|---|---|---|---|
| Snapshot cache | canonical worktree, checked against graph hash + indexed HEAD | tracked index and last path state | incompatible graph base or failed identity fence |
| Delta cache | exact `SnapshotIdentity` | extracted graph delta and shadow set | any identity change |
| Generation exact hit | exact `SnapshotIdentity` | immutable visible files, selector indexes, adjacency, remap | identity change / LRU eviction |
| Compatible generation seed | graph/worktree compatibility key | previous persistent generation for structural sharing | incompatible base/worktree |
| Generation singleflight | exact identity | one concurrent builder publishes to waiters | builder failure triggers retry |
| Request replay | one MCP request | repeated base/exact-fallback graph operations | end of request |
| Pinned generation | one MCP request | same immutable generation for every nested query | end of request |

There is deliberately **no TTL**. Exact identity, compatible-key seeding, LRU capacity, and singleflight are the right primitives.

`OverlayGeneration::update` rebuilds only changed file segments, affected stable-ID owners, rewritten selector slots, remap entries, and the changed-endpoint adjacency closure. Warm generation queries do not repeat shadow filtering, base/overlay merge, stable-ID deduplication, or result sorting.

## Cross-request journey: why subsequence calls do not merge

A single tool handler is pinned, so nested operations inside `code_subgraph`, `code_callers`, or `code_read_symbol` see one immutable generation. But a normal agent journey is multiple independent MCP requests:

`code_symbol_search → code_read_symbol → code_callers → code_callees`

The generation object is reused, yet each request independently establishes a start/end identity lease. The query cost is effectively zero; the validation lease is paid again. This is why “warm generation” does **not** imply sub-10-ms end-to-end `code_*`.

In [ ]:
sequenceDiagram
    participant Client
    participant Router
    participant Observer
    participant Generation
    Note over Client,Generation: @spec CROSS-REQUEST-JOURNEY
    Client->>Router: search_request
    Note over Client,Router: @message SEARCH_REQ<br/>@from Client<br/>@to Router<br/>@event search_request<br/>@order 1<br/>@when true<br/>@ensures SEARCH_STARTED: true
    Router->>Observer: search_start_observe
    Note over Router,Observer: @message SEARCH_START<br/>@from Router<br/>@to Observer<br/>@event search_start_observe<br/>@order 2<br/>@when true<br/>@ensures SEARCH_START_CERTIFIED: true
    Observer->>Generation: search_pin
    Note over Observer,Generation: @message SEARCH_PIN<br/>@from Observer<br/>@to Generation<br/>@event search_pin<br/>@order 3<br/>@when true<br/>@ensures SEARCH_PINNED: true
    Generation->>Router: search_result
    Note over Generation,Router: @message SEARCH_RESULT<br/>@from Generation<br/>@to Router<br/>@event search_result<br/>@order 4<br/>@when true<br/>@ensures SEARCH_RESULT_READY: true
    Router->>Observer: search_end_observe
    Note over Router,Observer: @message SEARCH_END<br/>@from Router<br/>@to Observer<br/>@event search_end_observe<br/>@order 5<br/>@when true<br/>@ensures SEARCH_END_CERTIFIED: true
    Router->>Client: search_response
    Note over Router,Client: @message SEARCH_RESPONSE<br/>@from Router<br/>@to Client<br/>@event search_response<br/>@order 6<br/>@when true<br/>@ensures SEARCH_RESPONDED: true
    Client->>Router: read_request
    Note over Client,Router: @message READ_REQ<br/>@from Client<br/>@to Router<br/>@event read_request<br/>@order 7<br/>@when true<br/>@ensures READ_STARTED: true
    Router->>Observer: read_start_observe
    Note over Router,Observer: @message READ_START<br/>@from Router<br/>@to Observer<br/>@event read_start_observe<br/>@order 8<br/>@when true<br/>@ensures READ_START_CERTIFIED: true
    Observer->>Generation: read_pin
    Note over Observer,Generation: @message READ_PIN<br/>@from Observer<br/>@to Generation<br/>@event read_pin<br/>@order 9<br/>@when true<br/>@ensures READ_PINNED: true
    Generation->>Router: read_result
    Note over Generation,Router: @message READ_RESULT<br/>@from Generation<br/>@to Router<br/>@event read_result<br/>@order 10<br/>@when true<br/>@ensures READ_RESULT_READY: true
    Router->>Observer: read_end_observe
    Note over Router,Observer: @message READ_END<br/>@from Router<br/>@to Observer<br/>@event read_end_observe<br/>@order 11<br/>@when true<br/>@ensures READ_END_CERTIFIED: true
    Router->>Client: read_response
    Note over Router,Client: @message READ_RESPONSE<br/>@from Router<br/>@to Client<br/>@event read_response<br/>@order 12<br/>@when true<br/>@ensures READ_RESPONDED: true
    Note over Client,Generation: @verify PROTOCOL: prove sequence_protocol

## Race and failure behavior

The primary route is fail-closed:

- start/end identity mismatch: discard candidate generation result, rebuild an exact delta, rerun the handler;
- snapshot command failure or malformed status: fall back to the filesystem oracle;
- one unstable snapshot: retry once, then filesystem oracle;
- generation build failure or budget expiration: exact request-scoped overlay;
- direct overlay budget expiration: legacy path may serve base only with stale/rebuild metadata.

Fallback is correct but expensive. It also re-enters `GraphResponseMetadata::analyze_source_inner`, so response-file OID work returns on exceptional paths.

In [ ]:
sequenceDiagram
    participant Router
    participant Generation
    participant Observer
    participant ExactOverlay
    participant Metadata
    participant Client
    Note over Router,Client: @spec IDENTITY-MISMATCH-FALLBACK
    Router->>Generation: pinned_query
    Note over Router,Generation: @message PINNED_QUERY<br/>@from Router<br/>@to Generation<br/>@event pinned_query<br/>@order 1<br/>@when true<br/>@ensures QUERY_EXECUTED: true
    Generation->>Router: candidate_result
    Note over Generation,Router: @message CANDIDATE<br/>@from Generation<br/>@to Router<br/>@event candidate_result<br/>@order 2<br/>@when true<br/>@ensures CANDIDATE_READY: true
    Router->>Observer: end_identity_check
    Note over Router,Observer: @message END_CHECK<br/>@from Router<br/>@to Observer<br/>@event end_identity_check<br/>@order 3<br/>@when true<br/>@ensures CHECK_STARTED: true
    Observer->>Router: identity_mismatch
    Note over Observer,Router: @message MISMATCH<br/>@from Observer<br/>@to Router<br/>@event identity_mismatch<br/>@order 4<br/>@when true<br/>@ensures MISMATCH_FOUND: true
    Router->>ExactOverlay: rebuild_exact_delta
    Note over Router,ExactOverlay: @message REBUILD<br/>@from Router<br/>@to ExactOverlay<br/>@event rebuild_exact_delta<br/>@order 5<br/>@when true<br/>@ensures DELTA_REBUILT: true
    ExactOverlay->>Router: exact_query_result
    Note over ExactOverlay,Router: @message EXACT_RESULT<br/>@from ExactOverlay<br/>@to Router<br/>@event exact_query_result<br/>@order 6<br/>@when true<br/>@ensures EXACT_READY: true
    Router->>Metadata: analyze_response_files
    Note over Router,Metadata: @message ANALYZE<br/>@from Router<br/>@to Metadata<br/>@event analyze_response_files<br/>@order 7<br/>@when true<br/>@ensures RESPONSE_SCANNED: true
    Metadata->>Client: fallback_response
    Note over Metadata,Client: @message FALLBACK_RESPONSE<br/>@from Metadata<br/>@to Client<br/>@event fallback_response<br/>@order 8<br/>@when true<br/>@ensures RESPONDED: true
    Note over Router,Client: @verify PROTOCOL: prove sequence_protocol

## Measured latency decomposition

### Deterministic 30-run fixture matrix

| Fixture | Direct Parquet p50/p95 | Warm generation query p50/p95 | Exact overlay oracle p50/p95 | Full MCP p50/p95 |
|---|---:|---:|---:|---:|
| Small, untracked-heavy | 0.095 / 0.198 ms | 0.00042 / 0.00063 ms | 45.77 / 48.70 ms | 94.20 / 97.38 ms |
| Medium, dirty Rust | 0.118 / 0.217 ms | 0.00071 / 0.00092 ms | 45.64 / 52.69 ms | 94.93 / 96.69 ms |
| Large, mostly-clean polyglot | 0.143 / 0.247 ms | 0.00217 / 0.00250 ms | 41.36 / 44.09 ms | 96.84 / 100.47 ms |

Matrix verdict: release-safe, exact-observation route, `complete_warm_under_10ms=false`, blocker `no_supported_synchronous_git_token_fence`.

### Current repository probes

- 10 exact porcelain-v2 status calls: 0.37 s total ≈ **37 ms/call**
- 20 `rev-parse HEAD` calls: 0.08 s total ≈ **4 ms/call**
- 10 indexed-HEAD diffs: 0.06 s total ≈ **6 ms/call**
- Six live warm MCP samples:
  - `code_symbol_search`: 200–206 ms after first, median ≈ **204 ms**
  - valid `code_resolve`: 408–433 ms after first, median ≈ **418 ms**
  - `code_read_symbol`: 405–424 ms after first, median ≈ **408 ms**

The live wall time includes MCP transport/server overhead and an unknown running-binary SHA. The isolated Git measurements explain the source-level floor: two snapshots × (`rev + status + rev`) are already about **90 ms** on this repository before tool transport.

## Architecture and logic-flow findings

| ID | Severity | Finding | Consequence |
|---|---|---|---|
| F1 | High | **Route split:** search uses legacy preflight; eight tools enter direct overlay before preflight. | Different latency, diagnostics, and failure behavior for the most common discovery tool. |
| F2 | High | **Lease scope is one MCP request, not one journey.** | Subsequent `search → read → callers` calls repeat all exact fences although the generation is unchanged. |
| F3 | High | **Validation dominates:** six Git subprocesses on the generic happy path; exact status enumerates all untracked files with acceleration disabled. | Warm generation cannot make individual requests sub-10-ms; large/untracked repositories amplify latency. |
| F4 | Medium | **Clean identity is represented as `None`.** The no-changed-path branch serves the base and calls `analyze_source_inner`. | Certified snapshot information is discarded; Git metadata and response-file OID work reappear. |
| F5 | Medium | **Metadata semantic violation:** `from_certified_overlay` forces `worktree_dirty=false`. | A response can report old indexed HEAD, newer worktree HEAD, and `worktree_dirty=false`; freshness is being confused with physical Git dirtiness. |
| F6 | Medium | **Argument validation occurs after backend open and overlay preparation** for generic handlers. | Invalid requests can pay the full observer/generation cost; observed missing-selector calls took roughly 400 ms. |
| F7 | Medium | **Diagnostics are route-local, not operation-derived.** Generation diagnostics hard-code two observations. | Search’s preceding legacy Git preflight is not represented, so performance telemetry can undercount actual Git work. |
| F8 | Low | Two Git-status implementations remain: legacy porcelain-v1/default config and exact porcelain-v2/fail-closed config. | Duplicate parsing and semantics increase maintenance risk and make timing comparisons ambiguous. |

### What does *not* violate the architecture

- Exact identity keys, compatible-generation seeding, LRU, and singleflight are sound.
- Persistent visible-file, selector, remap, and adjacency structures correctly move merge/sort/dedup work to generation update time.
- Query-time pinning prevents mixed generations inside one MCP request.
- Post-query identity mismatch discards the candidate result.
- Filesystem fallback preserves correctness when Git observation is unavailable.

No current evidence shows the primary dirty-path returning mixed-generation data. The problems are route uniformity, contract semantics, and lease cost.

## Recommended target architecture

### Priority 0 — correctness/contract cleanup

1. Preserve physical `worktree_dirty`; add a separate `response_fresh` / `overlay_certified` field.
2. Carry a certified identity even when the changed set is empty. Do not encode “identity overlay” as `None`.
3. Parse and validate tool arguments before backend open or Git observation.
4. Derive diagnostics from counted observer/subprocess events.

### Priority 1 — unify the family

5. Route `code_symbol_search` through the same direct generation wrapper as the other eight tools.
6. Delete or quarantine the legacy porcelain-v1 preflight from the Auto happy path.
7. Add route-parity tests asserting the same observer count, metadata semantics, and fallback behavior for all nine tools.

### Priority 2 — reduce the safe per-request floor

8. Use porcelain-v2 `--branch` and bind `branch.oid` into the observation, then prove that one pre-`rev-parse` can be removed while retaining the post fence. This saves two subprocesses per request, but status remains dominant.
9. Keep native fsmonitor/untracked cache only behind capability proof and exact fallback.

### Priority 3 — optimize the journey, not the individual call

10. Add a bounded `code_batch` / journey request that executes multiple typed operations against one pinned generation with one start/end fence.
11. Do **not** share a TTL lease across arbitrary requests. Without a synchronous Git token/barrier, an async watcher cannot prove absence of a change between notification and response.
12. Individual sub-10-ms requests require a supported synchronous token or in-process exact status parity; batching is the best proven solution available now.

### Required gates

- all-tool route parity;
- clean snapshot path with zero response scans;
- physical dirty metadata truth table;
- invalid args cause zero Git subprocesses;
- diagnostics equal actual observer/subprocess counts;
- batch result equals independent exact oracle;
- mutation between batch start/end forces full discard/fallback;
- 30-run small/medium/large matrix, cold and warm separated.

### Target-state architecture: reader view

This view translates the priorities above into runtime seams:

- **P0:** typed validation precedes every backend or Git observation; the certified identity lease always produces explicit freshness, physical dirty state, and counted diagnostics.
- **P1:** all nine `code_*` operations share one route; the legacy search preflight is absent.
- **P2:** porcelain-v2 branch identity and a post fence remain authoritative. Fsmonitor is only an optional accelerator and exact fallback remains mandatory.
- **P3:** a bounded journey batch shares one identity fence and one pinned immutable generation across multiple typed operations.

The required route-parity, mutation, oracle-equivalence, and performance-matrix gates remain verification boundaries rather than runtime services. The first diagram is deliberately one straight seven-stage spine for fast reading; the following verification view retains every detailed service and edge. Both use closed `@ns-policy` bindings without inventing security, capacity, or deployment claims.

In [ ]:
architecture-beta
    group public(cloud)[Caller]
    group edge(cloud)[MCP request control]
    group private(cloud)[Certified graph view]

    service client(internet)[MCP client] in public
    service control(server)[Route validate batch] in edge
    service identity(server)[Exact Git identity] in private
    service generation(database)[Overlay generation] in private
    service pinned(server)[Pinned view] in private
    service operations(server)[Nine code tools] in private
    service response(server)[Certified response] in private

    client:R --> L:control
    control:R --> L:identity
    identity:R --> L:generation
    generation:R --> L:pinned
    pinned:R --> L:operations
    operations:R --> L:response
    %% @ns-policy {"version":1,"services":[{"id":"client"},{"id":"control"},{"id":"identity"},{"id":"generation"},{"id":"pinned"},{"id":"operations"},{"id":"response"}],"edges":[{"from":"client","to":"control","protocol":"mcp","purpose":"request"},{"from":"control","to":"identity","protocol":"in_process","purpose":"route_validate_and_bound_journey"},{"from":"identity","to":"generation","protocol":"in_process","purpose":"certify_exact_identity"},{"from":"generation","to":"pinned","protocol":"in_process","purpose":"pin_immutable_generation"},{"from":"pinned","to":"operations","protocol":"in_process","purpose":"execute_code_family"},{"from":"operations","to":"response","protocol":"in_process","purpose":"certify_response"}]}

#### Verification view: complete seam topology

The compact reader view collapses three implementation clusters: **Route validate batch** contains the unified router, typed validation, and bounded journey; **Exact Git identity** contains the identity lease, porcelain-v2 observation, post fence, fsmonitor hint, exact fallback, and counted diagnostics; **Overlay generation** combines the stable graph base with the persistent delta generation. The detailed diagram below expands those clusters and retains the full 14-service, 18-edge, 51-obligation proof.

In [ ]:
architecture-beta
    group public(cloud)[Caller]
    group edge(cloud)[Unified MCP edge]
    group private(cloud)[Certified graph and worktree]

    service client(internet)[MCP client] in public

    service router(server)[Unified code router P1] in edge
    service validator(server)[Typed argument validation P0] in edge
    service batch(server)[Bounded journey batch P3] in edge

    service base(database)[Stable graph base] in private
    service lease(server)[Certified identity lease P0] in private
    service observer(server)[Status v2 and post fence P2] in private
    service accelerator(server)[Optional fsmonitor acceleration P2] in private
    service fallback(server)[Exact fallback authority] in private
    service generation(database)[Persistent overlay generation] in private
    service pinned(server)[Pinned generation] in private
    service operations(server)[All nine code operations P1] in private
    service metadata(server)[Freshness and dirty metadata P0] in private
    service diagnostics(server)[Counted observer diagnostics P0] in private

    client:R --> L:router
    router:R --> L:validator
    validator:B --> T:lease
    validator:R --> L:batch
    batch:B --> T:lease
    batch:R --> L:pinned
    lease:R --> L:base
    lease:B --> T:observer
    observer:R --> L:accelerator
    observer:R --> L:fallback
    observer:B --> T:generation
    accelerator:B --> T:generation
    fallback:B --> T:generation
    base:B --> T:generation
    generation:R --> L:pinned
    pinned:R --> L:operations
    operations:R --> L:metadata
    observer:R --> L:diagnostics
    %% @ns-policy {"version":1,"services":[{"id":"client"},{"id":"router"},{"id":"validator"},{"id":"batch"},{"id":"base"},{"id":"lease"},{"id":"observer"},{"id":"accelerator"},{"id":"fallback"},{"id":"generation"},{"id":"pinned"},{"id":"operations"},{"id":"metadata"},{"id":"diagnostics"}],"edges":[{"from":"client","to":"router","protocol":"mcp","purpose":"request"},{"from":"router","to":"validator","protocol":"in_process","purpose":"parse_arguments"},{"from":"validator","to":"lease","protocol":"in_process","purpose":"start_individual_request"},{"from":"validator","to":"batch","protocol":"in_process","purpose":"start_bounded_journey"},{"from":"batch","to":"lease","protocol":"in_process","purpose":"share_identity_fence"},{"from":"batch","to":"pinned","protocol":"in_process","purpose":"share_pinned_generation"},{"from":"lease","to":"base","protocol":"in_process","purpose":"bind_graph_base"},{"from":"lease","to":"observer","protocol":"in_process","purpose":"observe_exact_identity"},{"from":"observer","to":"accelerator","protocol":"in_process","purpose":"use_proven_fsmonitor"},{"from":"observer","to":"fallback","protocol":"in_process","purpose":"retain_exact_fallback"},{"from":"observer","to":"generation","protocol":"in_process","purpose":"publish_exact_identity"},{"from":"accelerator","to":"generation","protocol":"in_process","purpose":"accelerate_changed_set"},{"from":"fallback","to":"generation","protocol":"in_process","purpose":"rebuild_exact_delta"},{"from":"base","to":"generation","protocol":"in_process","purpose":"seed_generation"},{"from":"generation","to":"pinned","protocol":"in_process","purpose":"pin_immutable_generation"},{"from":"pinned","to":"operations","protocol":"in_process","purpose":"execute_code_family"},{"from":"operations","to":"metadata","protocol":"in_process","purpose":"certify_response"},{"from":"observer","to":"diagnostics","protocol":"in_process","purpose":"count_observer_events"}]}

## Source map

| Concern | Current source |
|---|---|
| MCP dispatch | `crates/spur-graph/src/mcp/mod.rs:115` — `GraphMcpModule::dispatch_current_project` |
| Search exception | `crates/spur-graph/src/mcp/mod.rs:1036` — `code_search_response` |
| Generic route | `crates/spur-graph/src/mcp/mod.rs:1506` — `code_graph_backend_response_with_refresh` |
| Overlay request lifecycle | `crates/spur-graph/src/mcp/mod.rs:1618` — `overlay_response_for_backend` |
| Worktree preparation | `crates/spur-graph/src/mcp/mod.rs:2052` — `prepare_overlay_for_worktree` |
| Post-query fence | `crates/spur-graph/src/mcp/mod.rs:1865` — `authoritative_overlay_identity` |
| Certified metadata | `crates/spur-graph/src/mcp/mod.rs:3572` — `GraphResponseMetadata::from_certified_overlay` |
| Legacy metadata scan | `crates/spur-graph/src/mcp/mod.rs:3633` — `GraphResponseMetadata::analyze_source_inner` |
| Snapshot algorithm | `crates/spur-graph/src/mcp/overlay_snapshot.rs:404` — `snapshot_with_runtime` |
| Candidate construction | `crates/spur-graph/src/mcp/overlay_snapshot.rs:662` — `build_snapshot_once` |
| Exact status route | `crates/spur-graph/src/git.rs:454` — `status_observation_with_runner` |
| Generation cache/singleflight | `crates/spur-graph/src/mcp/request_cache.rs:78` — `overlay_generation` |
| Generation LRU/compatible seed | `crates/spur-graph/src/mcp/request_cache.rs:531` — `OverlayGenerationCache` |
| Incremental generation | `crates/spur-graph/src/overlay_generation.rs:298` — `OverlayGeneration::update` |
| Warm query index | `crates/spur-graph/src/overlay_generation.rs:495` — `search_symbols_counted` |
| Request memoization | `crates/spur-graph/src/mcp/request_replay.rs:143` — `GraphQueryClient for RequestReplayClient` |
| Pinned query adapter | `crates/spur-graph/src/mcp/mod.rs:1257` — `GraphQueryClient for PinnedGenerationClient` |
| Benchmark matrix | `.spur/bench-evidence/task6-overlay-generation-matrix.json` |

### Formal-diagram scope

The NS-Mermaid sequence cells prove that the authored finite message contracts admit the declared ordering. The route flowchart proves its tool-class partition is total, deterministic, and exclusive. These proofs validate the notebook models; they do not substitute for Rust tests or prove the implementation refines the diagrams.